In [74]:
# Cell 1: Install required libraries
!pip install google-genai python-dotenv matplotlib pillow --quiet

print("✅ Library installation completed!")

✅ Library installation completed!


In [75]:
import tkinter as tk
from tkinter import scrolledtext
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import os
import threading
from PIL import Image, ImageTk
from dotenv import load_dotenv
from google import genai

# Load environment variables and API key
load_dotenv('data.env') 
API_KEY = os.getenv("GEMINI_API_KEY") 
client = genai.Client(api_key=API_KEY)

print("✅ Cell 1: Libraries and API are ready!")

✅ Cell 1: Libraries and API are ready!


In [76]:
try:
    df = pd.read_csv('sales_data.csv')
except FileNotFoundError:
    df = pd.DataFrame({
        'Month': [1, 2, 3, 4, 5],
        'Category': ['Electronics', 'Furniture', 'Electronics', 'Fashion', 'Electronics'],
        'Revenue': [1200, 800, 1500, 600, 1300],
        'Quantity': [5, 2, 7, 10, 6],
        'Marketing_Spend': [200, 150, 250, 100, 210],
        'High_Profit': [1, 0, 1, 0, 1]
    })

class RevenuePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, x):
        return self.linear(x)

def train_model():
    X = torch.tensor(df['Month'].values.reshape(-1, 1), dtype=torch.float32)
    y = torch.tensor(df['Revenue'].values.reshape(-1, 1), dtype=torch.float32)
    model = RevenuePredictor()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    for _ in range(1500):
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
    return model

pytorch_model = train_model()

print("✅ Cell 2: Data loaded and PyTorch model trained successfully!")

✅ Cell 2: Data loaded and PyTorch model trained successfully!


In [77]:
# Cell 3: AI and data processing functions (HYBRID AI - SIÊU MƯỢT)

def analyze_data(user_name="Boss"):
    plt.figure(figsize=(7, 4))
    plt.bar(df['Month'], df['Revenue'], color='skyblue', alpha=0.8)
    plt.plot(df['Month'], df['Revenue'], color='red', marker='o', linewidth=2)
    plt.title('Monthly Revenue Performance')
    plt.xlabel('Month')
    plt.ylabel('Revenue ($)')
    plt.grid(True, alpha=0.3)
    img_path = 'revenue_analysis.png'
    plt.savefig(img_path, bbox_inches='tight')
    plt.close() 
    
    msg = f"📊 Analysis completed for you, {user_name}!\nTotal Revenue: ${df['Revenue'].sum():,}\nAverage: ${df['Revenue'].mean():.2f}"
    return msg, img_path

def predict_revenue(user_name="Boss", next_month: int = 13):
    X_next = torch.tensor([[next_month]], dtype=torch.float32)
    pred = pytorch_model(X_next).item()
    plt.figure(figsize=(7, 4))
    plt.plot(df['Month'], df['Revenue'], 'bo-', label='Actual')
    plt.scatter([next_month], [pred], color='red', s=120, label='Prediction')
    plt.title(f'PyTorch Prediction - Month {next_month}')
    plt.xlabel('Month')
    plt.ylabel('Revenue ($)')
    plt.legend()
    plt.grid(True)
    img_path = 'pytorch_prediction.png'
    plt.savefig(img_path, bbox_inches='tight')
    plt.close()
    
    msg = f"🔮 PyTorch prediction completed for {user_name}!\nPredicted revenue for month {next_month}: ${pred:.2f}"
    return msg, img_path

def classify_sales(user_name="Boss"):
    avg = df['Revenue'].mean()
    temp_df = df.copy()
    temp_df['Status'] = temp_df['Revenue'].apply(
        lambda x: 'High' if x >= avg*1.1 else ('Low' if x <= avg*0.9 else 'Medium')
    )
    high_count = (temp_df['Status'] == 'High').sum()
    medium_count = (temp_df['Status'] == 'Medium').sum()
    low_count = (temp_df['Status'] == 'Low').sum()
    
    table_lines = ["Month | Category     | Revenue  | Status"]
    table_lines.append("-" * 44)
    for _, row in temp_df.iterrows():
        m = int(row['Month'])
        c = str(row['Category'])
        r = int(row['Revenue'])
        s = str(row['Status'])
        table_lines.append(f"{m:<5} | {c:<12} | ${r:<7} | {s}")
    table_str = "\n".join(table_lines)
    
    msg = f"📋 Classification ready, {user_name}!\n• High: {high_count} months\n• Medium: {medium_count} months\n• Low: {low_count} months\n\n📊 Detailed Table:\n{table_str}"
    return msg, None

def get_ai_response(user_query, user_name):
    q = user_query.lower().strip()
    
    # 1. TẦNG SIÊU TỐC
    if "analyze" in q or "chart" in q:
        return analyze_data(user_name)
    if "predict" in q or "forecast" in q:
        return predict_revenue(user_name)
    if "classify" in q or "table" in q:
        return classify_sales(user_name)

    # 2. TẦNG TRÍ TUỆ NHÂN TẠO
    tools_list = [analyze_data, predict_revenue, classify_sales]
    
    try:
        # Đã đổi cách viết prompt để KHÔNG BAO GIỜ bị lỗi Indentation nữa
        prompt = (
f"You are a highly intelligent and professional AI Assistant. Your manager's name is {user_name}. "
            "If the user asks for data analysis, prediction, or classification, you MUST call the appropriate tool. "
            "If the user says hello or asks general questions, respond naturally in English. "
            f"User says: '{user_query}'"
        )
        
        response = client.models.generate_content(
            model="gemini-pro",
            contents=prompt,
            config={'tools': tools_list}
        )
        
        if response.candidates and response.candidates[0].content.parts:
            part = response.candidates[0].content.parts[0]
            if part.function_call:
                func_name = part.function_call.name
                print(f"🧠 [AI Function Call] Executing: {func_name}()") 
                
                if func_name == "analyze_data":
                    return analyze_data(user_name)
                elif func_name == "predict_revenue":
                    return predict_revenue(user_name)
                elif func_name == "classify_sales":
                    return classify_sales(user_name)
                    
        return response.text, None 
        
    except Exception as e:
        print(f"API Error: {e}")
        return f"Sorry {user_name}, my AI core is currently unavailable due to network issues. ({e})", None

print("✅ Cell 3: Hybrid AI (Fast keywords + Function Calling) loaded!")

✅ Cell 3: Hybrid AI (Fast keywords + Function Calling) loaded!


In [78]:
class ManagerAssistantApp:
    def __init__(self, root):
        self.root = root
        self.root.title("✨ AI Manager Assistant Pro")
        self.root.geometry("650x750")
        self.root.configure(bg="#f0f2f5") 
        self.chat_images = [] 
        
        # Biến lưu trữ tên người dùng
        self.user_name = "" 
        
        chat_frame = tk.Frame(root, bg="#f0f2f5", bd=0)
        chat_frame.pack(padx=15, pady=15, fill=tk.BOTH, expand=True)
        
        self.chat_display = scrolledtext.ScrolledText(
            chat_frame, wrap=tk.WORD, state=tk.DISABLED, 
            font=("Consolas", 10), bg="white", fg="#1c1e21", 
            bd=0, padx=10, pady=10, relief="flat"
        )
        self.chat_display.pack(fill=tk.BOTH, expand=True)
        
        input_frame = tk.Frame(root, bg="#f0f2f5")
        input_frame.pack(padx=15, pady=(0, 10), fill=tk.X)
        
        self.user_input = tk.Entry(input_frame, font=("Segoe UI", 12), bg="white", bd=0, relief="flat")
        self.user_input.pack(side=tk.LEFT, fill=tk.X, expand=True, ipady=8, padx=(0, 10))
        self.user_input.bind("<Return>", lambda event: self.send_message())
        
        self.send_btn = tk.Button(
            input_frame, text="Send 🚀", font=("Segoe UI", 10, "bold"), 
            bg="#0866ff", fg="white", bd=0, activebackground="#0054d1", 
            activeforeground="white", width=10, cursor="hand2", command=self.send_message
        )
        self.send_btn.pack(side=tk.RIGHT, ipady=5)
        
        btn_frame = tk.Frame(root, bg="#f0f2f5")
        btn_frame.pack(padx=15, pady=(0, 15), fill=tk.X)
        btn_style = {"font": ("Segoe UI", 10, "bold"), "fg": "white", "bd": 0, "cursor": "hand2"}
        
        tk.Button(btn_frame, text="📊 Analyze", bg="#00a400", activebackground="#008c00", command=self.analyze_btn, **btn_style).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=3, ipady=6)
        tk.Button(btn_frame, text="🔮 Predict", bg="#8a2be2", activebackground="#7224b8", command=self.predict_btn, **btn_style).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=3, ipady=6)
        tk.Button(btn_frame, text="📋 Classify", bg="#fa383e", activebackground="#d62d33", command=self.classify_btn, **btn_style).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=3, ipady=6)
        
        # Lời chào đầu tiên đòi tên
        self.add_message("Assistant", "👋 Welcome to AI Manager Assistant!\n\n👉 Please type your NAME in the chat box to start:")

    def add_message(self, sender, message):
        self.chat_display.config(state=tk.NORMAL)
        self.chat_display.insert(tk.END, f"{sender}: ", "bold")
        self.chat_display.insert(tk.END, f"{message}\n\n")
        self.chat_display.config(state=tk.DISABLED)
        self.chat_display.yview(tk.END)
        
    def add_image_to_chat(self, image_path):
        try:
            img = Image.open(image_path)
            img = img.resize((450, 300), Image.Resampling.LANCZOS)
            photo = ImageTk.PhotoImage(img)
            self.chat_images.append(photo) 
            self.chat_display.config(state=tk.NORMAL)
            self.chat_display.image_create(tk.END, image=photo)
            self.chat_display.insert(tk.END, "\n\n")
            self.chat_display.config(state=tk.DISABLED)
            self.chat_display.yview(tk.END)
        except Exception as e:
            print(f"Image display error: {e}")

    def process_ai_request(self, query):
        text_resp, img_resp = get_ai_response(query, self.user_name)
        self.root.after(0, lambda: self.add_message("Assistant", text_resp))
        if img_resp:
            self.root.after(0, lambda: self.add_image_to_chat(img_resp))
            
    def send_message(self):
        user_text = self.user_input.get().strip()
        if not user_text: return
        
        self.user_input.delete(0, tk.END)
        self.add_message("You", user_text)
        
        # Nếu chưa có tên, lấy chữ vừa nhập làm tên luôn
        if not self.user_name:
            self.user_name = user_text
            welcome_msg = f"✨ Nice to meet you, {self.user_name}!\n\nThe AI system is fully operational. You can use the buttons above or type:\n1. 'analyze' (View charts)\n2. 'predict' (Next month forecast)\n3. 'classify' (Performance table)"
            self.add_message("Assistant", welcome_msg)
        else:
            threading.Thread(target=self.process_ai_request, args=(user_text,), daemon=True).start()

    def analyze_btn(self):
        if not self.user_name:
            self.add_message("System", "⚠️ Please type your name in the chat box first!")
            return
        text, img = analyze_data(self.user_name)
        self.add_message("Assistant", text)
        if img: self.add_image_to_chat(img)

    def predict_btn(self):
        if not self.user_name:
            self.add_message("System", "⚠️ Please type your name in the chat box first!")
            return
        text, img = predict_revenue(self.user_name)
        self.add_message("Assistant", text)
        if img: self.add_image_to_chat(img)

    def classify_btn(self):
        if not self.user_name:
            self.add_message("System", "⚠️ Please type your name in the chat box first!")
            return
        text, img = classify_sales(self.user_name)
        self.add_message("Assistant", text)

In [ ]:
if __name__ == "__main__":
    root = tk.Tk()
    app = ManagerAssistantApp(root)
    root.mainloop()

API Error: Failed to parse the parameter user_name='Boss' of function analyze_data for automatic function calling.Automatic function calling works best with simpler function signature schema, consider manually parsing your function declaration for function analyze_data.
